# 1.12 更新一次有什麼效果？


知道方向之後，還得真的動一步才能改善猜法。[1.11](https://birdhackor.github.io/tiny-perceptron-vlm/1.11.html) 顯示梯度與參數分開儲存；`backward()` 算出敏感度，不會替你把參數移到更好的位置。本節繼續一個旋鈕，刻意只更新一次，讓「求導」與「更新」兩個動作可以分別核對。

沿用代價 `(w-3)²`，w 從 1 開始，代價 4，梯度 -4。為了降低代價，我們沿梯度的反方向走：`新w = 舊w - 學習率×梯度`。學習率（learning rate）通常寫成 η，是每次步子的比例；η=0.1 時，`1-0.1×(-4)=1.4`。負梯度配上減號讓 w 增加，恰好往目標 3 靠近。新的代價為 `(1.4-3)²=2.56`。

這個方法叫梯度下降（gradient descent）：「下降」指希望代價降低，並不是所有參數都一定往小的方向改。把所有模型參數依各自梯度調一次，就是一次參數更新。多次用資料求導與更新才組成訓練，但我們先把這一個可驗算的動作做好。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

w = torch.tensor(1.0, requires_grad=True)
before = (w - 3).square()
before.backward()
print("更新前", w.item(), before.item(), w.grad.item())
with torch.no_grad():
    w -= 0.1 * w.grad
after = (w - 3).square()
print("更新後", w.item(), after.item())
assert after.item() < before.item()

輸入是一個要追蹤導數的 w。`before` 儲存更新前算出的代價，`before.backward()` 求出 -4。第一行輸出應為 `1,4,-4`。`w -= ...` 直接修改 w 本身，稱為原地更新；w 是我們直接建立且需追蹤梯度的來源參數，PyTorch 在一般求導模式下會拒絕這種修改，以免破壞求導所需的數值。若移除 `torch.no_grad()`，本例會在這一行立即以 `RuntimeError` 停下。在 `with torch.no_grad():` 的縮排範圍裡，則明確把這次更新排除在求導記錄之外，允許我們移動參數，再用新位置重新評估代價。離開這個縮排範圍後，w 仍需追蹤梯度，所以計算 `after` 時會建立新的求導關係。

`after` 必須用更新後的 w 重新計算，不能拿舊的 `before` 當新代價。第二行會顯示 w 約 1.4，代價約 2.56；浮點數可能顯示 `1.399999...`，這是有限精度，不影響結論。最後的檢查確認這次小步有改善，而非替任何學習率作保證。這段只做一次求導，因此沒有累積舊梯度的問題，下一節才會看重複求導。

若 η 太大，正確方向也可能跨過谷底再爬上另一邊。例如 η=2，讓 w 從 1 跳到 9，代價變成 36，比原來更差。反過來，η 極小則可能每步都改善卻要走很久。真正訓練時要同時看代價是否下降與變化是否穩定，不能只看「有梯度」就當成功。

練習先把步長 0.1 改成 0.6，手算新 w=3.4、代價 0.16，再執行核對，原檢查應通過。接著改成 2，先預測最後 `assert` 會失敗，再執行：它應該明確提醒代價沒有下降。這是有意安排的反例，幫你區分程式運作正確和設定是否合適。
